# Basic RAG chain

Query -> retrieve (pgvector, direct SQL) -> generate (LangChain LCEL + GPT-4o-mini).

Retrieves from the `chunks` table (OpenAI `text-embedding-3-small` embeddings) ingested by `ingest_pgvector.py` / `embed_missing.py`. Swap the table name to `chunks_nomic` to compare against local `nomic-embed-text` embeddings.

**Prerequisite:** `docker compose up -d` (Postgres+pgvector container `rag-postgres`).

In [1]:
import os

import psycopg
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

DSN = "host=localhost port=5432 dbname=rag user=rag password=rag"
TABLE = "chunks"  # or "chunks_nomic" to compare embedding models
EMB_MODEL = "text-embedding-3-small"

embeddings_model = OpenAIEmbeddings(model=EMB_MODEL, api_key=os.environ["OPENAI_KEY"])

## Retrieval

`chunks`/`chunks_nomic` have a custom flat schema (not LangChain's managed `PGVector` collection), so retrieval is a direct SQL query using pgvector's `<=>` cosine-distance operator, filtered optionally by `stock_symbol` and/or `section`.

In [2]:
def retrieve1_semantic(question: str, k: int = 5, stock_symbol: str = None, section: str = None) -> list[dict]:
    """Embed the question and return the top-k most similar chunks from Postgres."""
    query_embedding = embeddings_model.embed_query(question)
    embedding_str = "[" + ",".join(str(x) for x in query_embedding) + "]"

    where_clauses = []
    params = {"embedding": embedding_str, "k": k}
    if stock_symbol:
        where_clauses.append("stock_symbol = %(stock_symbol)s")
        params["stock_symbol"] = stock_symbol
    if section:
        where_clauses.append("section = %(section)s")
        params["section"] = section
    where_sql = f"WHERE {' AND '.join(where_clauses)}" if where_clauses else ""

    sql = f"""
        SELECT chunk_id, doc_id, stock_symbol, section, text,
               embedding <=> %(embedding)s::vector AS distance
        FROM {TABLE}
        {where_sql}
        ORDER BY distance
        LIMIT %(k)s
    """

    with psycopg.connect(DSN) as conn, conn.cursor(row_factory=psycopg.rows.dict_row) as cur:
        cur.execute(sql, params)
        return cur.fetchall()



def format_chunks(chunks: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[{c['section']} | {c['doc_id']}]\n{c['text']}" for c in chunks
    )

## Hybrid Search + RRF (reciprocal rank fusion)

Implement keyword search + semantic search combined with RRF

In [16]:


def sementic_search(question: str, params: dict, where_clauses: list[str]) -> list[dict]:
    """Embed the question and return the top-k most similar chunks from Postgres."""
    query_embedding = embeddings_model.embed_query(question)
    embedding_str = "[" + ",".join(str(x) for x in query_embedding) + "]"

    params["embedding"] = embedding_str
    where_sql = "WHERE " + " AND ".join(where_clauses) if where_clauses else ""
    sql = f"""
            SELECT chunk_id, doc_id, stock_symbol, section, text,
                embedding <=> %(embedding)s::vector AS distance
            FROM {TABLE}
            {where_sql}
            ORDER BY distance
            LIMIT %(k)s
        """

    with psycopg.connect(DSN) as conn, conn.cursor(row_factory=psycopg.rows.dict_row) as cur:
        cur.execute(sql, params)
        chunks = cur.fetchall()
    return chunks


def key_word_search(question: str, params: dict, where_clauses: list[str]) -> list[dict]:
    """Retrieve the top-k most relevant chunks using a keyword search."""   
    where_clauses.append("text @@@ paradedb.match('text', %(query)s)")
    params["query"] = question
    where_sql = "WHERE " + " AND ".join(where_clauses)

    sql = f"""
        SELECT chunk_id, doc_id, stock_symbol, section, text,
            paradedb.score(chunk_id) AS score
        FROM {TABLE}
        {where_sql}
        ORDER BY score DESC
        LIMIT %(k)s
    """

    with psycopg.connect(DSN) as conn, conn.cursor(row_factory=psycopg.rows.dict_row) as cur:
        cur.execute(sql, params)
        chunks = cur.fetchall()
    return chunks

def RRF(chunks1: list[dict], chunks2: list[dict], sementic_weight: float = 0.7, b: int = 2) -> list[dict]:
    """Reciprocal Rank Fusion (RRF) to combine two ranked lists of chunks
   Args:
       chunks1: List of chunks from semantic search
       chunks2: List of chunks from keyword search
       sementic_weight: Weight for the semantic search results
       b: Constant for the RRF formula
   Returns:
       List of combined and ranked chunks
   """
    scores = {}
    for rank, chunk in enumerate(chunks1):
        chunk_id = chunk["chunk_id"]
        scores[chunk_id] = scores.get(chunk_id, 0) + sementic_weight / (rank + b)

    for rank, chunk in enumerate(chunks2):
        chunk_id = chunk["chunk_id"]
        scores[chunk_id] = scores.get(chunk_id, 0) + (1 - sementic_weight) / (rank + b)
    print(f"scores: {scores}")
    # Combine chunks and sort by score
    combined_chunks = {chunk["chunk_id"]: chunk for chunk in chunks1 + chunks2}
    ranked_chunks = sorted(combined_chunks.values(), key=lambda x: scores[x["chunk_id"]], reverse=True)

    return ranked_chunks



def retrieve_hybrid(question: str, k: int = 10, stock_symbol: str = None, section: str = None) -> list[dict]:
    """Retrieve the top-k most relevant chunks using a hybrid search (BM25 + embeddings)."""

    where_clauses = []
    params = {"k": k}

    # meta data filtering
    if stock_symbol:
        where_clauses.append("stock_symbol = %(stock_symbol)s")
        params["stock_symbol"] = stock_symbol
    if section:
        where_clauses.append("section = %(section)s")
        params["section"] = section
    where_sql = f"WHERE {' AND '.join(where_clauses)}" if where_clauses else ""

    # semantic search using embeddings
    chunks1 = sementic_search(question, params, where_clauses)
    chunks2 = key_word_search(question, params, where_clauses)
    print(f"chunks1 chunk id and score: {[(chunk['chunk_id'], chunk['distance']) for chunk in chunks1]}")
    print(f"chunks2 chunk id and score: {[(chunk['chunk_id'], chunk['score']) for chunk in chunks2]}")
    ranked_chunks = RRF(chunks1, chunks2)
    return ranked_chunks[:k]


In [18]:
question = "What is the name of Boeing's segment that makes commercial jets, and what is its abbreviation??"
k = 5
stock_symbol = "ba"
chunks = retrieve_hybrid(question, k, stock_symbol=stock_symbol)



chunks1 chunk id and score: [('ba-20241231_section_1_chunk_1', 0.3078130039669469), ('ba-20241231_section_12_chunk_250', 0.45622177909357997), ('ba-20241231_section_12_chunk_18', 0.4668592551965305), ('ba-20241231_section_12_chunk_15', 0.48970572399485257), ('ba-20241231_section_10_chunk_35', 0.49178675618160117)]
chunks2 chunk id and score: [('ba-20241231_section_12_chunk_88', 21.214481), ('ba-20241231_section_12_chunk_22', 17.586086), ('ba-20241231_section_12_chunk_87', 15.6033125), ('ba-20241231_section_1_chunk_1', 15.001449), ('ba-20241231_section_12_chunk_235', 13.884714)]
scores: {'ba-20241231_section_1_chunk_1': 0.41, 'ba-20241231_section_12_chunk_250': 0.2333333333333333, 'ba-20241231_section_12_chunk_18': 0.175, 'ba-20241231_section_12_chunk_15': 0.13999999999999999, 'ba-20241231_section_10_chunk_35': 0.11666666666666665, 'ba-20241231_section_12_chunk_88': 0.15000000000000002, 'ba-20241231_section_12_chunk_22': 0.10000000000000002, 'ba-20241231_section_12_chunk_87': 0.07500000

## Generation chain (LCEL)

`retrieve` + `format_chunks` plug into a LangChain `Runnable` pipeline alongside the prompt and LLM.

In [ ]:
PROMPT_TEMPLATE = """\
You are a financial analyst assistant. Answer the question using only the provided excerpts \
from the company's 10-K annual filing. If the answer is not in the excerpts, say so.

Excerpts:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ["OPENAI_KEY"])


def make_rag_chain(k: int = 5, stock_symbol: str = None, section: str = None, retrieval_method: str = "hybrid"):
    """Build an LCEL RAG chain: retrieve -> format -> prompt -> llm -> parse."""

    def retrieve_and_format(question: str) -> str:
        if retrieval_method == "hybrid":
            chunks = retrieve_hybrid(question, k=k, stock_symbol=stock_symbol, section=section)
        else:
            chunks = retrieve1_semantic(question, k=k, stock_symbol=stock_symbol, section=section)
        return format_chunks(chunks)

    return (
        {"context": RunnableLambda(retrieve_and_format), "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )



## Ask questions

Ticker is one of: aapl, amd, avgo, ba, brka, cost, crsp, dpz, goog, ibm, intc, isrg, meta, mu, nvda, smr, snps, tsla, unh.

In [4]:
chain = make_rag_chain(k=5, stock_symbol="aapl")

answer = chain.invoke("What are Apple's main products and services?")
print(answer)

Apple's main products include smartphones (iPhone), personal computers (Mac), multipurpose tablets (iPad), wearables (smartwatches, wireless headphones, and spatial computers), and accessories. 

Apple's main services include advertising services, AppleCare (service and support products), cloud services, digital content (such as the App Store and subscription-based services like Apple Arcade, Apple Fitness+, Apple Music, Apple News+, and Apple TV), and payment services (Apple Card and Apple Pay).


In [5]:
# Inspect retrieved chunks to understand what the LLM saw
chunks = retrieve("What are the main risk factors?", k=5, stock_symbol="aapl")

for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} | {c['section']} | {c['doc_id']} | distance={c['distance']:.4f} ---")
    print(c["text"][:400])
    print()

--- Chunk 1 | Item 1A. | aapl-20250927 | distance=0.6242 ---
Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Companys business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this section are based on the Companys beliefs and opinions regarding matters that could materially adverse

--- Chunk 2 | Item 7A. | aapl-20250927 | distance=0.6974 ---
Item 7A. Quantitative and Qualitative Disclosures About Market Risk
The Company is exposed to economic risk from interest rates and foreign exchange rates. The Company uses various strategies to manage these risks; however, they may still impact the Companys consolidated financial statements.
Interest Rate Risk
The Company is primarily exposed to fluctuations in U.S. interest rates and their impac

--- Chunk 3 | Item 1A. | aapl-20250927 | distance=0.7087 ---
The Companys 

## Retrieval evaluation: semantic-only vs hybrid

Retrieval-only comparison on `eval/eval_set.jsonl` (34 hand-labeled questions). No LLM calls.

**Metric:** doc-level recall@k. For each question, `recall = |gold doc_ids found in top-k chunks| / |gold doc_ids|`. Multi-document questions need every gold doc in the top-k to score 1.0. Section labels are not checked, because their formatting varies across filings.

**Limits:** 34 questions is a small sample, so treat small differences as noise.

In [ ]:
import contextlib
import io
import json
from collections import defaultdict

K = 5  # top-k for both methods

with open("eval/eval_set.jsonl") as f:  # run from the project root
    eval_rows = [json.loads(line) for line in f]


def doc_recall(chunks: list[dict], gold_doc_ids: list[str]) -> float:
    """Fraction of gold doc_ids that appear among the retrieved chunks."""
    retrieved = {c["doc_id"] for c in chunks}
    gold = set(gold_doc_ids)
    return len(retrieved & gold) / len(gold)


def run_eval(retriever, k: int) -> list[dict]:
    results = []
    for row in eval_rows:
        with contextlib.redirect_stdout(io.StringIO()):  # retrieve_hybrid prints debug output
            chunks = retriever(row["question"], k=k)
        results.append({
            "id": row["id"],
            "category": row["category"],
            "recall": doc_recall(chunks, row["doc_ids"]),
        })
    return results


def summarize(results: list[dict]) -> dict[str, float]:
    by_cat = defaultdict(list)
    for r in results:
        by_cat[r["category"]].append(r["recall"])
    by_cat["overall"] = [r["recall"] for r in results]
    return {cat: sum(vals) / len(vals) for cat, vals in by_cat.items()}


semantic_results = run_eval(retrieve1_semantic, K)
hybrid_results = run_eval(retrieve_hybrid, K)

semantic_summary = summarize(semantic_results)
hybrid_summary = summarize(hybrid_results)

print(f"Doc-level recall@{K}")
print(f"{'category':<20}{'semantic':>10}{'hybrid':>10}{'diff':>10}")
for cat in ["factual_lookup", "multi_document", "difficult_retrieval", "synthesis", "overall"]:
    s, h = semantic_summary[cat], hybrid_summary[cat]
    print(f"{cat:<20}{s:>10.3f}{h:>10.3f}{h - s:>+10.3f}")

print("\nQuestions where the two methods differ:")
for s_row, h_row in zip(semantic_results, hybrid_results):
    if s_row["recall"] != h_row["recall"]:
        print(f"{s_row['id']} ({s_row['category']}): semantic={s_row['recall']:.2f} hybrid={h_row['recall']:.2f}")
